In [11]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import count, max
session = SparkSession.builder.getOrCreate()
apps_path = "apps.txt"
users_path = "users.txt"
actions_path = "actions.txt"

# Ex1
def check_year(timestamp:str):
    return True if timestamp.split("/")[0] == '2021' else False

def get_month(timestamp:str):
    return timestamp.split("/")[1]

session.udf.register("get_month", get_month)
session.udf.register("check_year", check_year)
apps = session.read.load(apps_path, format = 'csv', header = True, inferSchema = True)
actions = session.read.load(actions_path, format = 'csv', header = True, inferSchema = True)
actions = actions.selectExpr("AppId", "get_month(Timestamp) as Month", "Action").where("check_year(Timestamp) = True")
installations_count = actions.where("Action='Install'").groupby("AppID", "Month").agg(count("AppID").alias("NumbInstallations"))
removal_count = actions.where("Action='Remove'").groupby("AppID", "Month").agg(count("AppID").alias("NumbRemoval"))
joined_DF = installations_count.join(removal_count, ["AppID", "Month"], "full").na.fill(0, "NumbRemoval")
filteredDF = joined_DF.where("NumbInstallations > NumbRemoval")
finalDF = filteredDF.groupBy("AppID").agg(count("Month").alias("NoMonths")).where("NoMonths = 12")
finalDF = finalDF.join(apps, "AppID").select("AppID","AppName")
finalDF.show(10)

#Ex2
actions = session.read.load(actions_path, format = 'csv', header = True, inferSchema = True)
apps = session.read.load(apps_path, format = 'csv', header = True, inferSchema = True)
actions = actions.where("Action = 'Install'")
before_31 = actions.where("Timestamp <= '2021/12/31-23:59:59'")
after_31 = actions.where("Timestamp > '2021/12/31-23:59:59'")
desired_users = after_31.join(before_31, ["UserID", "AppId"], "left_anti")
desired_users = desired_users.groupby("AppID").agg(count("UserID").alias("NumberOfUsers"))
max_user = desired_users.groupby("AppID").agg(max("NumberOfUsers").alias("Max")).select("AppID")
max_user.show(10)                                                



26/01/13 21:17:53 WARN SimpleFunctionRegistry: The function get_month replaced a previously registered function.
26/01/13 21:17:53 WARN SimpleFunctionRegistry: The function check_year replaced a previously registered function.


+------+-------+
| AppID|AppName|
+------+-------+
|C1app1|   app1|
|C1app3|   app3|
+------+-------+

+------+
| AppID|
+------+
|C2app4|
|C2app3|
|C2app2|
+------+

